<a href="https://colab.research.google.com/github/Taqip/PB-AMALI_THAQIF/blob/main/DKB3263_ANN_MNIST_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DKB3263 – ANN untuk Klasifikasi MNIST

Notebook ini membina **Artificial Neural Network (ANN)** menggunakan **PyTorch** untuk mengklasifikasikan digit MNIST (0–9).

Struktur model:
- Input layer: 28 × 28 = 784 input
- 1 hidden layer: 128 neuron
- Activation: ReLU
- Output layer: 10 kelas
- Loss: CrossEntropyLoss
- Optimizer: Adam

Hyperparameter tuning:
- Sebelum: learning rate = 0.01
- Selepas: learning rate = 0.001


## 1. Import library dan pilih device

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device digunakan:", device)


Device digunakan: cuda


## 2. Load dan preprocess dataset MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Jumlah data latihan:", len(train_dataset))
print("Jumlah data ujian:", len(test_dataset))


## 3. Bina model ANN

In [ ]:
class SimpleANN(nn.Module):
    def __init__(self):
        super(SimpleANN, self).__init__()

        self.input_layer = nn.Linear(28 * 28, 128)
        self.relu = nn.ReLU()
        self.output_layer = nn.Linear(128, 10)

    def forward(self, x):
        # Tukar imej 28x28 kepada vektor 784
        x = x.view(x.size(0), -1)

        x = self.input_layer(x)
        x = self.relu(x)
        x = self.output_layer(x)

        return x

model = SimpleANN().to(device)
print(model)


## 4. Fungsi testing / validation

In [ ]:
def evaluate_model(model, data_loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy


## 5. Fungsi training

In [ ]:
def run_experiment(learning_rate, epochs=5):
    model = SimpleANN().to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    loss_history = []
    accuracy_history = []

    for epoch in range(epochs):
        model.train()

        running_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            # Reset gradient
            optimizer.zero_grad()

            # Forward pass
            outputs = model(images)

            # Kira loss
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()

            # Update weight
            optimizer.step()

            running_loss += loss.item()

        average_loss = running_loss / len(train_loader)
        test_accuracy = evaluate_model(model, test_loader)

        loss_history.append(average_loss)
        accuracy_history.append(test_accuracy)

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Loss: {average_loss:.4f} | "
            f"Test Accuracy: {test_accuracy:.2f}%"
        )

    return model, loss_history, accuracy_history


## 6. Eksperimen pertama – sebelum hyperparameter tuning

Learning rate awal: **0.01**


In [ ]:
baseline_lr = 0.01

baseline_model, baseline_loss, baseline_accuracy = run_experiment(
    learning_rate=baseline_lr,
    epochs=5
)

print("\nAccuracy akhir sebelum tuning:", f"{baseline_accuracy[-1]:.2f}%")


## 7. Eksperimen kedua – selepas hyperparameter tuning

Learning rate ditukar kepada **0.001**.


In [ ]:
tuned_lr = 0.001

tuned_model, tuned_loss, tuned_accuracy = run_experiment(
    learning_rate=tuned_lr,
    epochs=5
)

print("\nAccuracy akhir selepas tuning:", f"{tuned_accuracy[-1]:.2f}%")


## 8. Plot graf loss vs epoch

In [ ]:
epochs_range = range(1, len(baseline_loss) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, baseline_loss, marker="o", label="LR = 0.01")
plt.plot(epochs_range, tuned_loss, marker="o", label="LR = 0.001")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss vs Epoch")
plt.legend()
plt.grid(True)
plt.show()


## 9. Plot graf accuracy vs epoch

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(epochs_range, baseline_accuracy, marker="o", label="LR = 0.01")
plt.plot(epochs_range, tuned_accuracy, marker="o", label="LR = 0.001")

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy vs Epoch")
plt.legend()
plt.grid(True)
plt.show()


## 10. Perbandingan hasil

In [ ]:
print("===== PERBANDINGAN HYPERPARAMETER =====")
print(f"Sebelum tuning | Learning rate: {baseline_lr} | Accuracy: {baseline_accuracy[-1]:.2f}%")
print(f"Selepas tuning | Learning rate: {tuned_lr} | Accuracy: {tuned_accuracy[-1]:.2f}%")

difference = tuned_accuracy[-1] - baseline_accuracy[-1]

if difference > 0:
    print(f"Accuracy meningkat sebanyak {difference:.2f}% selepas tuning.")
elif difference < 0:
    print(f"Accuracy menurun sebanyak {abs(difference):.2f}% selepas tuning.")
else:
    print("Tiada perubahan pada accuracy selepas tuning.")


## 11. Uji satu sampel imej

Bahagian ini tidak wajib, tetapi sesuai untuk menunjukkan model boleh membuat ramalan.


In [ ]:
image, true_label = test_dataset[0]

tuned_model.eval()

with torch.no_grad():
    output = tuned_model(image.unsqueeze(0).to(device))
    predicted_label = output.argmax(dim=1).item()

plt.imshow(image.squeeze(), cmap="gray")
plt.title(f"Label sebenar: {true_label} | Ramalan: {predicted_label}")
plt.axis("off")
plt.show()


## Cara simpan notebook ini ke GitHub dari Google Colab

1. Cipta repository baru di GitHub, contoh: `DKB3263-ANN-MNIST`.
2. Buka notebook ini di Google Colab.
3. Jalankan semua cell dari atas ke bawah.
4. Klik **File > Save a copy in GitHub**.
5. Benarkan Google Colab mengakses GitHub jika diminta.
6. Pilih repository `DKB3263-ANN-MNIST`.
7. Pilih branch `main`.
8. Nama fail: `DKB3263_ANN_MNIST.ipynb`.
9. Tekan **OK**.

Cadangan fail dalam repository:
- `DKB3263_ANN_MNIST.ipynb`
- `README.md`
